In [1]:
import pandas as pd
from tqdm import tqdm
from itables import show as table_show

In [2]:
ENTREPOT_PATH = '~/Bureau/utils/data/'
METEO_PATH = '~/Bureau/utils/data/meteo/'
SPATIAL_PATH = './data/spatial/'

In [ ]:
df = {}

def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_entrepot = [
    'recolte_rendement_prix',
    'recolte_rendement_prix_restructure',
    'sdc', 
    'destination_valorisation',
    'action_realise',
    'action_realise_agrege',
    'action_synthetise_agrege',
    'composant_culture',
    'espece'
]

tables_performances = []

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_entrepot, ENTREPOT_PATH, sep = ',',index_col='id',verbose=False) 
import_dfs(tables_performances, ENTREPOT_PATH, sep = ',',verbose=False) 

100%|██████████| 9/9 [00:27<00:00,  3.03s/it]
0it [00:00, ?it/s]


In [35]:
studied_sdc_ids = ['fr.inra.agrosyst.api.entities.GrowingSystem_b92c0e13-c2ea-4144-ae7a-eb24b8d730fb',
       'fr.inra.agrosyst.api.entities.GrowingSystem_9d7fadfb-3354-433a-921c-5058d2a2a8bb',
       'fr.inra.agrosyst.api.entities.GrowingSystem_27f7d820-55e9-457e-bf59-e9934e36da2e',
       'fr.inra.agrosyst.api.entities.GrowingSystem_44d12e6e-5ff9-4808-b499-0718c5751ff0',
       'fr.inra.agrosyst.api.entities.GrowingSystem_9bb0f5ba-0f39-42a6-aa9a-a8436f267bf7',
       'fr.inra.agrosyst.api.entities.GrowingSystem_fd7d487b-1bb1-4404-9ead-a3cd424ac50e',
       'fr.inra.agrosyst.api.entities.GrowingSystem_165bc553-7038-4d1c-bc76-130f514621bc']

In [36]:
FILIERE = "VITICULTURE"

In [37]:
# ajout des informations nécessaires à recolte_rendement_prix
left = df['recolte_rendement_prix']
right = pd.concat([df['action_realise_agrege'], df['action_synthetise_agrege']])
df['recolte_rendement_prix_extanded'] = pd.merge(left, right, left_on = 'action_id', right_index=True, how='left')

# ajout des informations nécessaires à recolte_rendement_prix
left = df['recolte_rendement_prix_extanded']
right = df['sdc'][['filiere']]
df['recolte_rendement_prix_extanded'] = pd.merge(left, right, left_on = 'sdc_id', right_index=True, how='left')


In [55]:
df['recolte_rendement_prix_test'] = df['recolte_rendement_prix'].loc[
    df['recolte_rendement_prix_extanded']['sdc_id'].isin(studied_sdc_ids)
]
df['recolte_rendement_prix_restructure_test'] = df['recolte_rendement_prix_restructure'].loc[
    df['recolte_rendement_prix_restructure'].index.isin(df['recolte_rendement_prix_test'].index)
]
df['sdc_test'] = df['sdc'].loc[
    df['sdc'].index.isin(studied_sdc_ids)
]
df['action_realise_test'] = df['action_realise'].loc[
    df['action_realise'].index.isin(df['recolte_rendement_prix_test']['action_id'])
]
df['action_realise_agrege_test'] = df['action_realise_agrege'].loc[
    df['action_realise_agrege'].index.isin(df['action_realise_test'].index)
]
df['composant_culture_test'] = df['composant_culture'].loc[
    df['composant_culture'].index.isin(df['recolte_rendement_prix_restructure_test']['composant_culture_id'])
]
df['espece_test'] = df['espece'].loc[
    df['espece'].index.isin(df['composant_culture_test']['espece_id'])
]

In [56]:
path='./'
df['recolte_rendement_prix_test'].to_csv(path+'recolte_rendement_prix.csv')
df['recolte_rendement_prix_restructure_test'].to_csv(path+'recolte_rendement_prix_restructure'+'.csv')
df['sdc_test'].to_csv(path+'sdc'+'.csv')
df['action_realise_test'].to_csv(path+'action_realise'+'.csv')
df['action_realise_agrege_test'].to_csv(path+'action_realise_agrege'+'.csv')
df['composant_culture_test'].to_csv(path+'composant_culture'+'.csv')
df['espece_test'].to_csv(path+'espece'+'.csv')